## **Prepare csv training data for Inferno**

In [3]:
import torch
import warnings
import pandas as pd
from pathlib import Path
from tqdm import tqdm

from torch.utils.data import DataLoader

from config import OUT_DIR
from src.Dataset import ChestXRayDataset
from src.Model import InfernoCalibNet

from tqdm import TqdmExperimentalWarning

warnings.filterwarnings("ignore", category=TqdmExperimentalWarning)


def extract_id(image_path: str) -> str:
    return Path(image_path).stem


def infernoDataPrep():
    torch.cuda.empty_cache()

    test_dt = ChestXRayDataset(OUT_DIR / "test.csv", transform=False, return_aux=True)
    test_loader = DataLoader(
        test_dt, batch_size=32, shuffle=False, num_workers=4, pin_memory=True
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = InfernoCalibNet(num_classes=3).to(device)

    model.load_state_dict(
        torch.load(
            OUT_DIR / "InfernoCalibNet_model.pth",
            map_location=device,
            weights_only=True,  # Explicitly set to only load weights
        )
    )
    model.eval()

    results = []

    with torch.no_grad():
        for images, labels, aux_data in tqdm(test_loader, desc="Processing", leave=False):
            images = images.to(device, non_blocking=True)
            outputs = model(images)
            logits = outputs.cpu().numpy()

            for i in range(len(images)):
                image_id = extract_id(aux_data['IMGPATH'][i])
                
                # Map gender: 'F' -> 1, 'M' -> 0, otherwise None
                gender = aux_data.get('GENDER', [None])[i]
                gender = 1 if gender == 'F' else (0 if gender == 'M' else None)

                # Map VP: 'PA' -> 1, 'AP' -> 0, otherwise None
                vp = aux_data.get('VP', [None])[i]
                vp = 1 if vp == 'PA' else (0 if vp == 'AP' else None)

                true_label = labels[i].item()  # Extract true label

                results.append([image_id, logits[i][0], logits[i][1], logits[i][2], gender, vp, true_label])

    df = pd.DataFrame(results, columns=["ID", "L1", "L2", "L3", "GENDER", "VP", "TL"])
    df.to_csv(OUT_DIR / "infernoTrain.csv", index=False)

    return df


# infernoTrain
df_results = infernoDataPrep()
print("Datapoints in csv for Inferno training: ", len(df_results))


Datapoints in csv for Inferno training:  1512
